# Car Price Prediction — Fixed & Improved

Что исправлено:
- `condition` не используется и удаляется, если есть.
- Нет target encoding leakage: split делается до обучения/препроцессинга.
- Фильтр `model_count >= MIN_MODEL_COUNT` применяется только на train dataset, не на manual predict.
- `generation` теперь ожидается как код (`XV70`, `J150`, `F15`, `W212/S212`, `unknown`).
- CatBoost работает с категориальными признаками напрямую, без `LabelEncoder`, поэтому нет ошибки `unseen labels`.
- Manual predict не удаляется из-за фильтров.
- Диапазон цены считается через процент, а не через огромный std по всем машинам.

In [12]:
import pandas as pd
import numpy as np
import re
import pickle
from pathlib import Path

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CURRENT_YEAR = 2026
RANDOM_STATE = 42

MIN_MODEL_COUNT = 50       # если хочешь только популярные модели: 50/100. Если хочешь все — поставь 1.
RARE_CATEGORY_MIN_COUNT = 10


DATA_PATHS = [
    "cars_ml_10k.csv",
]

DATA_PATH = None
for p in DATA_PATHS:
    if Path(p).exists():
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError("CSV file not found. Put cars_ml_10k_generation_codes_no_condition.csv near this notebook.")

df_raw = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())

df_raw.head()

Loaded: cars_ml_10k.csv
Shape: (10542, 14)
Columns: ['brand', 'model', 'year', 'price', 'city', 'mileage_km', 'body_type', 'engine_volume_l', 'fuel_type', 'transmission', 'drive_type', 'steering_wheel', 'color', 'generation']


,brand,model,year,price,city,mileage_km,body_type,engine_volume_l,fuel_type,transmission,drive_type,steering_wheel,color,generation
0,Toyota,Highlander Luxe,2017,17590000,Астана,230186.0,Кроссовер,3.5,бензин,Автомат,Полный привод,Слева,черный,U5
1,Rox,01 VIP,2026,30490000,Алматы,NaN,Внедорожник,1.5,гибрид,Автомат,Полный привод,Слева,NaN,unknown
2,Hyundai,Elantra,2020,7700000,Алматы,183426.0,Седан,2.0,бензин,Автомат,Передний привод,Слева,серый металлик,AD/ADA
3,Kia,Sportage,2014,7000000,Петропавловск,114500.0,Кроссовер,2.0,бензин,Механика,Передний привод,Слева,серый,SL
4,Toyota,Land Cruiser Prado,2014,14999999,Актобе,NaN,Внедорожник,2.7,бензин,Автомат,Полный привод,Слева,NaN,J150


In [13]:
# Basic check

print("Missing values:")
print(df_raw.isna().sum().sort_values(ascending=False))

print("\nTop brands:")
print(df_raw["brand"].value_counts().head(15))

print("\nTop models:")
print(df_raw["model"].value_counts().head(15))

Missing values:
color              3729
mileage_km         3045
engine_volume_l     175
fuel_type             3
model                 2
transmission          1
drive_type            1
steering_wheel        1
brand                 0
year                  0
price                 0
city                  0
body_type             0
generation            0
dtype: int64

Top brands:
brand
Toyota           1847
Hyundai          1687
BMW              1521
ВАЗ               570
Kia               517
Mercedes-Benz     491
Lexus             313
Chevrolet         261
Volkswagen        252
Nissan            240
ГАЗ               205
Changan           193
Audi              184
BYD               167
Mitsubishi        162
Name: count, dtype: int64

Top models:
model
Camry                 664
X5                    328
Tucson                248
Accent                227
Elantra               226
Sonata                222
Grandeur              208
Land Cruiser Prado    166
Santa Fe              166
Land Cr

In [ ]:
CAT_COLUMNS = [
    "brand",
    "model",
    "city",
    "body_type",
    "fuel_type",
    "transmission",
    "drive_type",
    "steering_wheel",
    "color",
    "generation",   
]

NUM_FEATURES = [
    "year",
    "mileage_km",
    "engine_volume_l",
    "car_age",
    "mileage_per_year",
    "engine_volume_sq",
    "age_times_volume",
    "age_sq",
    "mileage_missing",
    "generation_missing",
    "generation_digits",
    "is_new",
    "is_luxury",
]

def normalize_generation_code(x):
    """
    Приводит generation к коду.
    Если в колонке уже XV70/J150/F15 — оставит код.
    Если там полный текст — попробует вытащить код.
    """
    if pd.isna(x):
        return "unknown"

    s = str(x).strip().upper()
    if s in ["", "NAN", "NONE", "UNKNOWN", "0"]:
        return "unknown"

    # Ищем коды вида XV70, J150, F15, W212/S212, G05, etc.
    codes = re.findall(r"\b[A-ZА-Я]{1,4}\d{1,4}(?:/[A-ZА-Я]{1,4}\d{1,4})*\b", s)
    if codes:
        return codes[-1]

    return s


def clean_base(df, is_train=True, min_model_count=MIN_MODEL_COUNT, popular_brands=POPULAR_BRANDS):
   
    df = df.copy()

    # Удаляем condition полностью.
    df = df.drop(columns=["condition"], errors="ignore")

    if "price" not in df.columns:
        df["price"] = 0

    # Обязательные колонки для стабильности.
    required_cols = [
        "brand", "model", "year", "price", "city", "mileage_km",
        "body_type", "engine_volume_l", "fuel_type", "transmission",
        "drive_type", "steering_wheel", "color", "generation"
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = np.nan

    # Типы.
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["engine_volume_l"] = pd.to_numeric(df["engine_volume_l"], errors="coerce")
    df["mileage_km"] = pd.to_numeric(df["mileage_km"], errors="coerce")

    # generation = только код.
    df["generation"] = df["generation"].apply(normalize_generation_code)

    if is_train:
        # Убираем плохие строки только при обучении.
        df = df.dropna(subset=["brand", "model", "price", "year", "engine_volume_l"])
        df = df[(df["price"] > 100_000) & (df["price"] < 200_000_000)]
        df = df[(df["year"] >= 1990) & (df["year"] <= CURRENT_YEAR)]
        df = df[df["engine_volume_l"] > 0]

        if popular_brands is not None:
            df = df[df["brand"].isin(popular_brands)]

        # Фильтр популярных моделей.
        # НЕЛЬЗЯ применять на manual predict.
        if min_model_count and min_model_count > 1:
            counts = df["model"].value_counts()
            good_models = counts[counts >= min_model_count].index
            df = df[df["model"].isin(good_models)]

    if len(df) == 0:
        raise ValueError("After cleaning dataframe has 0 rows. Check filters / input data.")

    return df.reset_index(drop=True)


def add_features(df, train_mileage_median=None):
    df = df.copy()

    # Mileage.
    df["mileage_missing"] = df["mileage_km"].isna().astype(int)

    if train_mileage_median is None:
        train_mileage_median = df["mileage_km"].median()
        if pd.isna(train_mileage_median):
            train_mileage_median = -1

    df["mileage_km"] = df["mileage_km"].fillna(train_mileage_median)

    # Age.
    df["car_age"] = (CURRENT_YEAR - df["year"]).clip(lower=0)

    # Numerical interactions.
    df["mileage_per_year"] = df["mileage_km"] / (df["car_age"] + 1)
    df["engine_volume_sq"] = df["engine_volume_l"] ** 2
    df["age_times_volume"] = df["car_age"] * df["engine_volume_l"]
    df["age_sq"] = df["car_age"] ** 2

    # Generation features.
    gen = df["generation"].fillna("unknown").astype(str)
    df["generation_missing"] = gen.str.lower().isin(["unknown", "nan", ""]).astype(int)

    df["generation_digits"] = gen.str.extract(r"(\d+)")[0]
    df["generation_digits"] = pd.to_numeric(df["generation_digits"], errors="coerce").fillna(-1)

    # Flags.
    df["is_new"] = ((df["year"] >= CURRENT_YEAR - 1) | (df["mileage_km"] == 0)).astype(int)
    df["is_luxury"] = df["brand"].isin(["Porsche", "BMW", "Mercedes-Benz", "Mercedes", "Audi", "Lexus"]).astype(int)

    return df


def fit_category_maps(df, cat_columns=CAT_COLUMNS, min_count=RARE_CATEGORY_MIN_COUNT):
    """
    Сохраняем допустимые категории только на train.
    Редкие категории идут в 'other'.
    Новые категории на predict тоже идут в 'other'.
    """
    maps = {}

    for col in cat_columns:
        if col in df.columns:
            s = df[col].fillna("unknown").astype(str)
            counts = s.value_counts()
            allowed = set(counts[counts >= min_count].index)
            allowed.add("unknown")
            allowed.add("other")
            maps[col] = allowed

    return maps


def apply_category_maps(df, category_maps):
    df = df.copy()

    for col, allowed in category_maps.items():
        if col not in df.columns:
            df[col] = "unknown"

        s = df[col].fillna("unknown").astype(str)
        s = s.where(s.isin(allowed), "other")
        df[col] = s

    return df


def make_X(df, category_maps, features=None, train_mileage_median=None):
    """
    Делает X для CatBoost.
    Категории остаются строками — CatBoost сам их обработает.
    """
    df = add_features(df, train_mileage_median=train_mileage_median)
    df = apply_category_maps(df, category_maps)

    use_cat = [c for c in CAT_COLUMNS if c in df.columns]
    use_num = [c for c in NUM_FEATURES if c in df.columns]

    X = df[use_cat + use_num].copy()

    for col in use_cat:
        X[col] = X[col].fillna("unknown").astype(str)

    for col in use_num:
        X[col] = pd.to_numeric(X[col], errors="coerce").fillna(-1)

    if features is not None:
        X = X.reindex(columns=features, fill_value=-1)
        for col in use_cat:
            if col in X.columns:
                X[col] = X[col].fillna("unknown").astype(str)

    return X

In [4]:
# ВАЖНО: сначала чистим raw data, потом делаем split.
# Так мы избегаем target leakage.

df_clean = clean_base(
    df_raw,
    is_train=True,
    min_model_count=MIN_MODEL_COUNT,
    popular_brands=POPULAR_BRANDS
)

print("After cleaning:", df_clean.shape)
print("Models left:", df_clean["model"].nunique())
print("Brands left:", df_clean["brand"].nunique())

train_df, val_df = train_test_split(
    df_clean,
    test_size=0.2,
    random_state=RANDOM_STATE
)

# Fit preprocessing only on train.
train_mileage_median = train_df["mileage_km"].median()
if pd.isna(train_mileage_median):
    train_mileage_median = -1

train_for_maps = add_features(train_df, train_mileage_median=train_mileage_median)
category_maps = fit_category_maps(train_for_maps)

X_train = make_X(
    train_df,
    category_maps=category_maps,
    train_mileage_median=train_mileage_median
)

features = X_train.columns.tolist()

X_val = make_X(
    val_df,
    category_maps=category_maps,
    features=features,
    train_mileage_median=train_mileage_median
)

y_train = np.log1p(train_df["price"])
y_val = np.log1p(val_df["price"])

cat_features = [col for col in CAT_COLUMNS if col in X_train.columns]

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Cat features:", cat_features)
print("Features:", features)

After cleaning: (4506, 14)
Models left: 36
Brands left: 10
Train: (3604, 23) (3604,)
Val: (902, 23) (902,)
Cat features: ['brand', 'model', 'city', 'body_type', 'fuel_type', 'transmission', 'drive_type', 'steering_wheel', 'color', 'generation']
Features: ['brand', 'model', 'city', 'body_type', 'fuel_type', 'transmission', 'drive_type', 'steering_wheel', 'color', 'generation', 'year', 'mileage_km', 'engine_volume_l', 'car_age', 'mileage_per_year', 'engine_volume_sq', 'age_times_volume', 'age_sq', 'mileage_missing', 'generation_missing', 'generation_digits', 'is_new', 'is_luxury']


In [5]:
# CatBoost training

cat_model = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=8,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    early_stopping_rounds=300,
    verbose=300,
    thread_count=-1
)

cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)

0:	learn: 0.8056004	test: 0.7695601	best: 0.7695601 (0)	total: 60.3ms	remaining: 5m 1s
300:	learn: 0.2151692	test: 0.2338177	best: 0.2338177 (300)	total: 737ms	remaining: 11.5s
600:	learn: 0.1995993	test: 0.2289075	best: 0.2289075 (600)	total: 1.4s	remaining: 10.2s
900:	learn: 0.1872991	test: 0.2259095	best: 0.2258804 (889)	total: 2.01s	remaining: 9.17s
1200:	learn: 0.1773028	test: 0.2236381	best: 0.2236381 (1200)	total: 2.64s	remaining: 8.36s
1500:	learn: 0.1694171	test: 0.2218593	best: 0.2218330 (1498)	total: 3.26s	remaining: 7.6s
1800:	learn: 0.1625721	test: 0.2207168	best: 0.2206668 (1771)	total: 3.9s	remaining: 6.93s
2100:	learn: 0.1557777	test: 0.2195997	best: 0.2195904 (2096)	total: 4.54s	remaining: 6.26s
2400:	learn: 0.1500237	test: 0.2190506	best: 0.2190481 (2383)	total: 5.17s	remaining: 5.59s
2700:	learn: 0.1450429	test: 0.2184031	best: 0.2183862 (2695)	total: 5.79s	remaining: 4.93s
3000:	learn: 0.1400458	test: 0.2178717	best: 0.2178717 (3000)	total: 6.42s	remaining: 4.28s
33

CatBoostRegressor(depth=6, early_stopping_rounds=300, iterations=5000, l2_leaf_reg=8, learning_rate=0.03, loss_function='RMSE', random_seed=42, verbose=300)

In [6]:
# Evaluation

pred_log = cat_model.predict(X_val)

mae_log = mean_absolute_error(y_val, pred_log)
rmse_log = np.sqrt(mean_squared_error(y_val, pred_log))
r2 = r2_score(y_val, pred_log)

real_price = np.expm1(y_val)
pred_price = np.expm1(pred_log)

ape = np.abs(pred_price - real_price) / real_price * 100
abs_error = np.abs(pred_price - real_price)

print("=== Metrics ===")
print(f"MAE log:  {mae_log:.4f}")
print(f"RMSE log: {rmse_log:.4f}")
print(f"R²:       {r2:.4f}")

print("\n=== Real price errors ===")
print(f"MAE тенге: {mean_absolute_error(real_price, pred_price):,.0f} ₸")
print(f"Median absolute error: {np.median(abs_error):,.0f} ₸")
print(f"Mean error %: {np.mean(ape):.2f}%")
print(f"Median error %: {np.median(ape):.2f}%")
print(f"80% cars error <= {np.percentile(ape, 80):.2f}%")
print(f"90% cars error <= {np.percentile(ape, 90):.2f}%")

=== Metrics ===
MAE log:  0.1420
RMSE log: 0.2169
R²:       0.9238

=== Real price errors ===
MAE тенге: 1,375,662 ₸
Median absolute error: 752,040 ₸
Mean error %: 14.88%
Median error %: 9.10%
80% cars error <= 20.01%
90% cars error <= 31.09%


In [7]:
# Error by brand/model.
# Это помогает понять, где модель ошибается сильнее.

result = val_df[["brand", "model", "year", "price"]].copy()
result["pred_price"] = pred_price
result["abs_error"] = abs_error
result["ape"] = ape

print("=== Error by brand ===")
display(
    result.groupby("brand")["ape"]
    .agg(["count", "median", "mean"])
    .sort_values("median", ascending=False)
    .head(20)
)

print("=== Error by model ===")
display(
    result.groupby("model")["ape"]
    .agg(["count", "median", "mean"])
    .sort_values("median", ascending=False)
    .head(20)
)

=== Error by brand ===


,count,median,mean
brand,,,
Daewoo,4,31.036217,65.343173
Volkswagen,36,26.749101,35.327286
ГАЗ,15,19.729825,25.189324
Toyota,259,10.909602,15.285839
BMW,166,10.122164,15.315390
ВАЗ,30,9.835042,18.050889
Subaru,15,9.813555,14.914945
Kia,55,7.879858,11.343195
Hyundai,299,7.032817,11.657203


=== Error by model ===


,count,median,mean
model,,,
Golf,12,29.722017,28.556883
Passat,17,29.335904,49.404043
Alphard,11,24.033995,28.280296
ГАЗель,15,19.729825,25.189324
Highlander,16,18.204093,15.847150
Sorento,14,14.873914,16.964255
X7,9,14.747096,13.295373
520,17,13.427580,24.396680
525,18,12.606605,23.095367


In [8]:
# Feature importance

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": cat_model.get_feature_importance()
}).sort_values("importance", ascending=False)

print("=== Top 20 important features ===")
display(importance.head(20))

=== Top 20 important features ===


,feature,importance
13,car_age,14.901110
10,year,11.773523
12,engine_volume_l,10.702983
15,engine_volume_sq,8.531547
17,age_sq,7.978019
6,drive_type,6.998366
5,transmission,6.902509
20,generation_digits,5.363563
3,body_type,4.769198
0,brand,4.329036


In [9]:
def predict_car_price(car_data, interval_pct=0.15):
    """
    Manual predict для одной машины.

    car_data: dict
    interval_pct=0.15 значит диапазон ±15%.
    """
    car_df = pd.DataFrame([car_data])

    # Никаких фильтров по model_count на predict.
    car_df = clean_base(car_df, is_train=False)

    X_car = make_X(
        car_df,
        category_maps=category_maps,
        features=features,
        train_mileage_median=train_mileage_median
    )

    pred_log = cat_model.predict(X_car)[0]
    pred_price = float(np.expm1(pred_log))

    low = pred_price * (1 - interval_pct)
    high = pred_price * (1 + interval_pct)

    return pred_price, low, high, X_car


# Тестовый пример.
# ВАЖНО: значения категорий должны быть похожи на train: бензин, Автомат, Передний привод, Слева, Седан.
test_car = {
    "brand": "Toyota",
    "model": "Camry",
    "year": 2021,
    "city": "Алматы",
    "mileage_km": 80000,
    "body_type": "Седан",
    "engine_volume_l": 2.5,
    "fuel_type": "бензин",
    "transmission": "Автомат",
    "drive_type": "Передний привод",
    "steering_wheel": "Слева",
    "color": "Серый",
    "generation": "XV70",
}

price, low, high, X_car_debug = predict_car_price(test_car, interval_pct=0.15)

print("\n=== Manual predict ===")
print(f"Машина: {test_car['brand']} {test_car['model']} {test_car['year']}")
print(f"Параметры: {test_car['engine_volume_l']}L, {test_car['mileage_km']:,} км, {test_car['generation']}")
print(f"Predicted price: {price:,.0f} ₸")
print(f"Range ±15%: {low:,.0f} - {high:,.0f} ₸")

print("\nX_car used by model:")
display(X_car_debug)


=== Manual predict ===
Машина: Toyota Camry 2021
Параметры: 2.5L, 80,000 км, XV70
Predicted price: 13,798,815 ₸
Range ±15%: 11,728,992 - 15,868,637 ₸

X_car used by model:


,brand,model,city,body_type,fuel_type,transmission,drive_type,steering_wheel,color,generation,...,car_age,mileage_per_year,engine_volume_sq,age_times_volume,age_sq,mileage_missing,generation_missing,generation_digits,is_new,is_luxury
0,Toyota,Camry,Алматы,Седан,бензин,other,Передний привод,Слева,other,XV70,...,5,13333.333333,6.25,12.5,25,0,0,70,0,0


In [10]:
# Save model and preprocessing metadata

MODEL_PATH = "car_price_catboost.cbm"
META_PATH = "car_price_preprocess.pkl"

cat_model.save_model(MODEL_PATH)

meta = {
    "features": features,
    "cat_features": cat_features,
    "category_maps": category_maps,
    "train_mileage_median": train_mileage_median,
    "current_year": CURRENT_YEAR,
    "min_model_count": MIN_MODEL_COUNT,
    "rare_category_min_count": RARE_CATEGORY_MIN_COUNT,
}

with open(META_PATH, "wb") as f:
    pickle.dump(meta, f)

print("Saved:", MODEL_PATH)
print("Saved:", META_PATH)

Saved: car_price_catboost.cbm
Saved: car_price_preprocess.pkl


In [ ]:
# How to load later

# from catboost import CatBoostRegressor
# import pickle
#
# loaded_model = CatBoostRegressor()
# loaded_model.load_model("car_price_catboost.cbm")
#
# with open("car_price_preprocess.pkl", "rb") as f:
#     loaded_meta = pickle.load(f)
#
# features = loaded_meta["features"]
# cat_features = loaded_meta["cat_features"]
# category_maps = loaded_meta["category_maps"]
# train_mileage_median = loaded_meta["train_mileage_median"]